# Elaboración Dataset Definitivo — TFG Guillermo Jiménez

Script de limpieza y estandarización del `Dataset_Definitivo.xlsx`.

**Pasos aplicados:**
1. Estandarización de columnas de jugadores (inglés → español)
2. Estandarización de tabs de equipos
3. Corrección de valores de mercado erróneos (cruce manual con Transfermarkt)
4. Eliminación de filas sin valor de mercado público

**Fuente:** `Dataset_Definitivo.xlsx` (output de Scraping_Lesiones_y_Valores_de_Mercado)  
**Destino:** `Dataset_Definitivo.xlsx` (sobreescribe con versión limpia)

In [1]:
!pip install -q openpyxl xlsxwriter pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.0 MB/s eta 0:00:00


## Imports y configuración

In [2]:
import pandas as pd
import numpy as np
import os

## Rutas

In [3]:
import os
DB = '/content'

SOURCE = os.path.join(DB, 'Dataset_Definitivo.xlsx')
DEST   = os.path.join(DB, 'Dataset_Definitivo.xlsx')

## Mapas de renombrado de columnas

In [4]:
PLAYER_RENAME = {
    'Player':            'Jugador',
    'Nation':            'Nacionalidad',
    'Pos':               'Posición',
    'Squad':             'Equipo',
    'Age':               'Edad',
    'Born':              'Año de nacimiento',
    'MP':                'Partidos jugados esta temporada',
    'Starts':            'Titularidades',
    'Min':               'Minutos jugados esta temporada',
    '90s':               'Minutos totales/90',
    'Gls':               'Goles',
    'Ast':               'Asistencias',
    'G+A':               'G+A',
    'G-PK':              'Goles que no son de penalti',
    'PK':                'Goles de penalti',
    'PKatt':             'Penaltis tirados',
    'CrdY':              'Amarillas',
    'CrdR':              'Rojas',
    'xG':                'Goles esperados',
    'npxG':              'Goles esperados sin contar penaltis',
    'xAG':               'Asistencias esperadas',
    'npxG+xAG':          'Contribuciones de gol esperadas sin penaltis',
    'PrgC':              'Conducción de balón de al menos 10 metros hacia la portería rival',
    'PrgP':              'Pases progresivos *',
    'PrgR':              'Recepciones de pases progresivos',
    'Gls.1':             'Goles/ 90 mins',
    'Ast.1':             'Asistencias/ 90mins',
    'G+A.1':             'G+A/ 90 mins',
    'G-PK.1':            'Goles sin contar penaltis/ 90 mins',
    'G+A-PK':            'G+A sin contar penaltis/ 90 mins',
    'xG.1':              'Goles esperados/ 90 mins',
    'xAG.1':             'Asistencias esperadas/ 90 mins',
    'xG+xAG':            '(Goles+Asistencias) esperados/ 90 mins',
    'npxG.1':            'Goles esperados sin contar penaltis/ 90 mins',
    'npxG+xAG.1':        '(Goles sin contar penaltis + Asistencias) esperados/ 90 mins',
    'market_value_eur':  'Valor de mercado',
    'dob_tm':            'Fecha de nacimiento',
    'tm_player_id':      'ID del jugador en Transfermarkt',
    'match_confidence':  'match_confidence',
    'contract_until':    'Fecha de expiración del contrato',
    'contract_years_remaining': 'Años de contrato restantes',
    'injury_count':      'Cantidad total de lesiones registradas',
    'injury_days_per_season': 'Promedio de días de baja por lesión por temporada',
    'injury_frequency':  'Número promerio de lesiones por temporada',
    'club_revenue_eur':  'Ingreso total anual del club',
    'revenue_tier':      'Categoría de ingresos del club**',
    'mv_history_points': '***Índice de madurez de carrera',
    'inj_days_last1':    'Días de lesión en la temporada pasada',
    'inj_days_avg_last2':'Promedio de días de lesión entre la temporada pasada y la anterior',
    'inj_days_avg_last3':'Promedio de días de lesión entre las 3 últimas temporadas',
    'inj_trend_slope':   'Pendiente de la regresión lineal de la cantidad de lesiones (días de lesión añadidos o disminuidos en promedio anual)',
    'inj_trend_pvalue':  'P-valor de la pendiente: Probabilidad de que la tendencia de la pendiente sea un ruido aleatorio (Aceptamos la hipótesis nula (pendiente válida) con un 10% o menos)',
    'inj_trend_sig':     'Aceptación o refutación del modelo (Uso un 10% en vez del 5% habitual por las pocas variables que hay en este análisis, para que no salga todo estable)',
    'inj_ewa_days':      'Media anual ponderada exponencialmente de los días de baja por lesión',
    'inj_risk':          'Riesgo de lesión',
    'inj_series_type':   'inj_series_type',
    'Entradas ganadas (FBref)':         'Entradas ganadas ',
    'Intercepciones (FBref)':           'Intercepciones',
    'Faltas cometidas (FBref)':         'Faltas cometidas ',
    'Faltas recibidas (FBref)':         'Faltas recibidas ',
    'Centros (FBref)':                  'Centros',
    'Tarjetas amarillas (FBref)':       'Tarjetas amarillas',
    'Tarjetas rojas (FBref)':           'Tarjetas rojas ',
    'Goles (FBref)':                    'Goles ',
    'Tiros totales (FBref)':            'Tiros totales',
    'Tiros a puerta (FBref)':           'Tiros a puerta ',
    'Precisión de tiro % (FBref)':      'Precisión de tiro % ',
    'Tiros por 90 min (FBref)':         'Tiros por 90 min',
    'Penaltis marcados (FBref)':        'Penaltis marcados',
    'Penaltis tirados (FBref)':         'Penaltis tirados.1',
    'PJ Portero (FBref)':               'PJ Portero ',
    'Titularidades Portero (FBref)':    'Titularidades Portero ',
    'Goles encajados/90 (FBref)':       'Goles encajados/90 ',
    'Tiros a puerta recibidos (FBref)': 'Tiros a puerta recibidos ',
    'Paradas (FBref)':                  'Paradas ',
    '% Paradas (FBref)':                '% Paradas ',
    'Porterías a cero (FBref)':         'Porterías a cero ',
    '% Porterías a cero (FBref)':       '% Porterías a cero ',
    'Penaltis recibidos (FBref)':       'Penaltis recibidos ',
    'Penaltis encajados (FBref)':       'Penaltis encajados ',
    'Penaltis parados (FBref)':         'Penaltis parados ',
    '% Penaltis parados (FBref)':       '% Penaltis parados ',
}

COLS_TO_DROP_NONPL = ['Unnamed: 2', 'Rk', 'market_value_raw', 'tm_name_matched', 'Matches']

TEAM_RENAME = {
    'Squad':            'Equipo',
    '# Pl':             'Cantidad de jugadores',
    'Age':              'Promedio de edad',
    'Poss':             'Promedio del porcentaje de posesión ',
    'Gls':              'Goles',
    'Ast':              'Asistencias',
    'G+A':              'Goles + Asistencias',
    'G-PK':             'Goles que no son de penalti',
    'PK':               'Goles de penalti',
    'PKatt':            'Penaltis tirados',
    'CrdY':             'Amarillas',
    'CrdR':             'Rojas',
    'xG':               'Goles esperados',
    'npxG':             'Goles esperados sin contar penaltis',
    'xAG':              'Asistencias esperadas',
    'npxG+xAG':         'Goles esperados sin penaltis + asistencias esperadas',
    'PrgC':             'Conducciones de balón significativas',
    'PrgP':             'Pases de avances significativos',
    'Gls_p90':          'Goles por partido',
    'Ast_p90':          'Asistencias por partido',
    'G+A_p90':          'Goles + Asistencias por partido',
    'G-PK_p90':         'Goles sin contar penaltis por partido',
    'G+A-PK':           'Goles sin penaltis + Asistencias por partido',
    'xG_p90':           'Goles esperados por partido',
    'xAG_p90':          'Asistencias esperadas por partido',
    'xG+xAG':           '(Goles + Asistencias) esperados por partido',
    'npxG_p90':         'Goles esperados por partido sin contar penaltis',
    'npxG+xAG_p90':     '(Goles sin contar penaltis + Asistencias) esperados por partido',
}

## Correcciones manuales de valores de mercado

Jugadores con datos cruzados incorrectamente en el matching fuzzy, corregidos con valores reales de Transfermarkt.

In [5]:
MV_FIXES = {
    'PL Players': {
        'Estêvão Willian': {'tm_id': 1056993, 'mv': 80_000_000},
        'Igor':            {'tm_id': None,     'mv': None},
        'Bradley Burrowes':{'tm_id': 1202361,  'mv':  2_000_000},
        'Joe Knight':      {'tm_id':  978668,  'mv':    100_000},
        'Rio Ngumoha':     {'tm_id': 1108466,  'mv': 20_000_000},
    },
    'LaLiga Players': {
        'Rodrigo Mendoza': {'tm_id':  961297,  'mv': 20_000_000},
        'Rubén':           {'tm_id':  705813,  'mv':  1_500_000},
        'Alexandre Alemão':{'tm_id':  560417,  'mv':  3_500_000},
        'Dawda Camara':    {'tm_id':  891923,  'mv':  1_000_000},
        'Sergio Lozano':   {'tm_id': 1384902,  'mv':     25_000},
    },
    'Serie A Players': {
        'Gabriele Piccinini': {'tm_id': 628365, 'mv': 4_000_000},
        'Massimo Pessina':    {'tm_id': 992569, 'mv': 1_000_000},
    },
    'Bundesliga Players': {
        'Wisdom Mike':   {'tm_id': 1084539, 'mv': 3_000_000},
        'Patrice Čović': {'tm_id': 1114155, 'mv': 4_000_000},
    },
    'Ligue 1 Players': {
        'Francisco Sierralta':      {'tm_id':  371436, 'mv':  1_500_000},
        'Telli Siwe':               {'tm_id': 1091353, 'mv':  1_000_000},
        'Daren Mosengo':            {'tm_id': 1119880, 'mv':     50_000},
        'Soriba Diaoune':           {'tm_id': 1184007, 'mv':    300_000},
        'Mathys De Carvalho':       {'tm_id': 1004486, 'mv':  4_000_000},
        'Ismaël Guerti':            {'tm_id':  860053, 'mv':    100_000},
        'Aladji Bamba':             {'tm_id':  975347, 'mv':  3_500_000},
        'Quentin Ndjantou Mbitcha': {'tm_id': 1053208, 'mv':  7_000_000},
        'Adama Camara':             {'tm_id':  973949, 'mv':  1_500_000},
        'Dayann Methalie':          {'tm_id': 1191444, 'mv': 12_000_000},
        'Raphael Le Guen':          {'tm_id': 1166473, 'mv':     50_000},
        'Stephan Zagadou':          {'tm_id': 1228317, 'mv':    100_000},
    },
}

## Funciones de procesamiento

In [9]:
def process_player_tab(df, tab_name, is_pl=False):
    """Estandariza un tab de jugadores al formato de PL Players.
    Funciona tanto con columnas en inglés (raw) como en español (ya procesadas)."""
    df = df.copy()
    df.dropna(how='all', inplace=True)

    # Detectar si las columnas ya están en español (archivo ya procesado)
    already_spanish = 'Jugador' in df.columns or 'Valor de mercado' in df.columns

    if not already_spanish:
        if is_pl:
            rename_map = {df.columns[2]: 'Jugador'}
            df.rename(columns=rename_map, inplace=True)
            drop_trailing = [c for c in df.columns if str(c).startswith('Unnamed: 8')]
            df.drop(columns=drop_trailing, inplace=True)
        else:
            drop = [c for c in COLS_TO_DROP_NONPL if c in df.columns]
            df.drop(columns=drop, inplace=True)

        # Insertar columna 'G+A sin contar penaltis' solo si no existe ya
        col_nueva = 'G+A sin contar penaltis'
        if (col_nueva not in df.columns
                and 'G+A' in df.columns
                and 'G-PK' in df.columns
                and 'Ast' in df.columns):
            try:
                ga_idx = df.columns.get_loc('G+A')
                # get_loc puede devolver int o slice/array si hay duplicados
                if not isinstance(ga_idx, int):
                    ga_idx = int(df.columns.tolist().index('G+A')) + 1
                else:
                    ga_idx = ga_idx + 1
                df.insert(ga_idx, col_nueva, df['G-PK'] + df['Ast'])
            except Exception:
                pass  # Si falla la inserción, continuamos sin esa columna

        df.rename(columns=PLAYER_RENAME, inplace=True)

    jugador_col = 'Jugador'
    mv_col = 'Valor de mercado'
    id_col = 'ID del jugador en Transfermarkt'
    conf_col = 'match_confidence'

    if tab_name in MV_FIXES and jugador_col in df.columns:
        for player, fix in MV_FIXES[tab_name].items():
            mask = df[jugador_col] == player
            if mask.any():
                if id_col in df.columns:
                    df.loc[mask, id_col] = fix['tm_id'] if fix['tm_id'] is not None else np.nan
                if mv_col in df.columns:
                    df.loc[mask, mv_col] = fix['mv'] if fix['mv'] is not None else np.nan
                if conf_col in df.columns:
                    df.loc[mask, conf_col] = np.nan
                print(f'  [{tab_name}] Corregido: {player}')

    if mv_col in df.columns:
        before = len(df)
        df = df[df[mv_col].notna()]
        dropped = before - len(df)
        if dropped:
            print(f'  [{tab_name}] Eliminadas {dropped} filas sin valor de mercado')

    df.reset_index(drop=True, inplace=True)
    return df


def find_header_row(df_raw):
    """Detecta la fila del encabezado buscando 'Squad' o 'Equipo'."""
    for i, row in df_raw.iterrows():
        if 'Squad' in row.values or 'Equipo' in row.values:
            return i
    return None


def process_team_tab(df_raw, tab_name):
    """Estandariza un tab de equipos al formato de PL Teams."""
    header_row = find_header_row(df_raw)
    if header_row is None:
        raise ValueError(f'No se encontró fila de encabezado en {tab_name}')

    header_vals = list(df_raw.iloc[header_row])
    data_rows = df_raw.iloc[header_row + 1:].copy()

    seen = {}
    deduped = []
    for h in header_vals:
        if pd.isna(h):
            deduped.append(None)
        else:
            h = str(h)
            if h in seen:
                seen[h] += 1
                deduped.append(f'{h}_p90')
            else:
                seen[h] = 1
                deduped.append(h)

    data_rows.columns = deduped

    # Filtrar filas vacías buscando la columna de equipo (Squad o Equipo)
    squad_col = 'Squad' if 'Squad' in data_rows.columns else 'Equipo'
    data_rows = data_rows[
        data_rows[squad_col].notna() &
        (data_rows[squad_col].astype(str).str.strip() != '')
    ]

    for c in ['MP', 'Starts', 'Min', '90s']:
        if c in data_rows.columns:
            data_rows.drop(columns=[c], inplace=True)

    data_rows.drop(columns=[c for c in data_rows.columns if c is None], inplace=True)
    data_rows.rename(columns=TEAM_RENAME, inplace=True)
    data_rows.reset_index(drop=True, inplace=True)
    return data_rows

## Ejecución: limpieza y guardado

In [12]:
print('Cargando Dataset_Definitivo.xlsx...')
raw_sheets = pd.read_excel(SOURCE, sheet_name=None, header=0)

print('Hojas disponibles:', list(raw_sheets.keys()))

PLAYER_TABS = [
    'PL Players', 'LaLiga Players', 'Serie A Players',
    'Bundesliga Players', 'Ligue 1 Players'
]
TEAM_TABS = [
    'LaLiga Teams', 'Serie A Teams',
    'Bundesliga Teams', 'Ligue 1 Teams'
]

all_dfs = {}

print('\n=== Tabs de jugadores ===')
for tab in PLAYER_TABS:
    if tab not in raw_sheets:
        print(f'  AVISO: hoja "{tab}" no encontrada, omitiendo')
        continue
    df_raw = raw_sheets[tab]
    is_pl  = (tab == 'PL Players')
    df_clean = process_player_tab(df_raw, tab, is_pl)
    print(f'  {tab}: {df_raw.shape[0]} filas → {df_clean.shape[0]} filas')
    all_dfs[tab] = df_clean

print('\n=== Tabs de equipos ===')
if 'PL Teams' in raw_sheets:
    all_dfs['PL Teams'] = raw_sheets['PL Teams']
    print(f'  PL Teams: {raw_sheets["PL Teams"].shape} (sin cambios)')
else:
    print('  AVISO: hoja "PL Teams" no encontrada en raw_sheets')

for tab in TEAM_TABS:
    if tab not in raw_sheets:
        print(f'  AVISO: hoja "{tab}" no encontrada, omitiendo')
        continue
    df_raw   = pd.read_excel(SOURCE, sheet_name=tab, header=None)
    df_clean = process_team_tab(df_raw, tab)
    print(f'  {tab}: {df_clean.shape}')
    all_dfs[tab] = df_clean

for tab in ['MV History', 'Injuries by Season']:
    if tab in raw_sheets:
        all_dfs[tab] = raw_sheets[tab]

print(f'\nGuardando en {DEST}...')
sheet_order = PLAYER_TABS + ['PL Teams'] + TEAM_TABS + ['MV History', 'Injuries by Season']
with pd.ExcelWriter(DEST, engine='xlsxwriter') as writer:
    for sheet in sheet_order:
        if sheet in all_dfs:
            all_dfs[sheet].to_excel(writer, sheet_name=sheet, index=False)


print('\nResumen de filas:')
for tab in PLAYER_TABS:
    if tab in all_dfs:
        print(f'  {tab}: {len(all_dfs[tab])} jugadores')

Cargando Dataset_Definitivo.xlsx...
Hojas disponibles: ['PL Players', 'LaLiga Players', 'Serie A Players', 'Bundesliga Players', 'Ligue 1 Players', 'MV History', 'Injuries by Season']

=== Tabs de jugadores ===
  [PL Players] Corregido: Estêvão Willian
  [PL Players] Corregido: Bradley Burrowes
  [PL Players] Corregido: Joe Knight
  [PL Players] Corregido: Rio Ngumoha
  PL Players: 415 filas → 415 filas
  [LaLiga Players] Corregido: Rodrigo Mendoza
  [LaLiga Players] Corregido: Rubén
  [LaLiga Players] Corregido: Alexandre Alemão
  [LaLiga Players] Corregido: Dawda Camara
  [LaLiga Players] Corregido: Sergio Lozano
  LaLiga Players: 384 filas → 384 filas
  [Serie A Players] Corregido: Gabriele Piccinini
  [Serie A Players] Corregido: Massimo Pessina
  Serie A Players: 375 filas → 375 filas
  [Bundesliga Players] Corregido: Wisdom Mike
  [Bundesliga Players] Corregido: Patrice Čović
  Bundesliga Players: 335 filas → 335 filas
  [Ligue 1 Players] Corregido: Francisco Sierralta
  [Ligue 1